# Notebook 03 — Sequential Pattern Mining (Taxi-Day Sequences)

**Project:** Urban Taxi Mobility Pattern Discovery (Microsoft T-Drive)

**Objective.** Build one sequence per **taxi-day** (one taxi, one calendar
date), mine frequent ordered sub-sequences of visited 500m grid cells with
bounded PrefixSpan, and report a support-sensitivity, time-segment, and
weekday/weekend analysis of the results.

**Methodology constraints preserved in this notebook (do not change):**
- Sequence unit: **one taxi on one calendar day** (a "taxi-day"). Calendar
  days are never concatenated into a weekly sequence.
- Spatial grid: **500m**, from `spatial.assign_spatial_cells()`.
- Sequential mining algorithm: **PrefixSpan** (ordered, non-contiguous
  subsequences) — not n-grams.
- `MAX_PATTERN_LENGTH = 4` (spatial cells, not taxis) on every PrefixSpan
  call.
- Support = (# taxi-day sequences containing the pattern) / (**original**
  total # taxi-day sequences). The denominator never changes because of
  support-pruning, and it is never the number of GPS points or distinct
  taxis.
- Minimum support **count** is computed with `math.ceil(min_support * n)`,
  not `round()` — `round()` can produce a count that fails the stated
  `support >= min_support` requirement (see Section 6 of the correction
  request; e.g. `round(0.05 * 446) = 22`, but `22 / 446 = 4.93% < 5%`).


## 1. Imports and project path setup

In [1]:
import math
import sys
from pathlib import Path

import pandas as pd

# The project's own src/ package must be importable BEFORE any
# similarly-named installed package (this was previously an actual bug:
# an installed `sequential` package from site-packages shadowed
# src/sequential.py). insert(0, ...) — not append() — guarantees this.
PROJECT_ROOT = Path.cwd().parent if (Path.cwd() / "notebooks").exists() is False else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_DIR))

import config
import spatial
import sequential

print("sequential loaded from:", sequential.__file__)
print("spatial loaded from:", spatial.__file__)
assert str(SRC_DIR) in sequential.__file__, (
    "sequential was NOT loaded from the project's src/ directory — "
    "check sys.path ordering and for a same-named installed package."
)


sequential loaded from: c:\Users\user\Downloads\urban-taxi-mobility\urban-taxi-mobility\src\sequential.py
spatial loaded from: c:\Users\user\Downloads\urban-taxi-mobility\urban-taxi-mobility\src\spatial.py


## 2. Load processed data

Loads the cleaned, labeled GPS data produced by Notebook 02:
`data/processed/labeled_sample.csv`.


In [2]:
PROCESSED_DATA_FILENAME = "labeled_sample.csv"
processed_path = config.PROCESSED_DATA_DIR / PROCESSED_DATA_FILENAME

if processed_path.exists():
    traj_df = pd.read_csv(processed_path, parse_dates=[config.COL_TIMESTAMP])
else:
    candidates = sorted(config.PROCESSED_DATA_DIR.glob("*"))
    raise FileNotFoundError(
        f"Expected processed data at {processed_path}, which does not exist.\n"
        f"Files actually present in {config.PROCESSED_DATA_DIR}:\n"
        + "\n".join(f"  - {p.name}" for p in candidates)
    )

print(f"Loaded {len(traj_df):,} GPS rows for {traj_df[config.COL_TAXI_ID].nunique():,} taxis.")
traj_df.head()


Loaded 116,913 GPS rows for 99 taxis.


,taxi_id,timestamp,lon,lat,in_region,time_diff_sec,distance_m,speed_kmh,is_suspicious_jump,spatial_cell_id,cluster_id,hour,time_segment,day_type
0,10134,2008-02-02 13:42:35,116.35761,39.90600,True,NaN,NaN,NaN,False,CELL_01727,0,13,afternoon,weekend
1,10134,2008-02-02 13:52:38,116.34042,39.91870,True,603.0,2035.624366,12.152981,False,CELL_01873,0,13,afternoon,weekend
2,10134,2008-02-02 13:55:15,116.34028,39.91757,True,157.0,126.216250,2.894131,False,CELL_01873,0,13,afternoon,weekend
3,10134,2008-02-02 14:00:35,116.34021,39.91747,True,320.0,12.620689,0.141983,False,CELL_01873,0,14,afternoon,weekend
4,10134,2008-02-02 14:15:48,116.34125,39.92185,True,913.0,495.043537,1.951979,False,CELL_01952,0,14,afternoon,weekend


## 3. Validate data

In [3]:
required_columns = {config.COL_TAXI_ID, config.COL_TIMESTAMP, config.COL_LAT, config.COL_LON}
missing = required_columns - set(traj_df.columns)
assert not missing, f"Processed data is missing required columns: {missing}"

assert pd.api.types.is_datetime64_any_dtype(traj_df[config.COL_TIMESTAMP]), (
    "timestamp column must already be parsed to datetime by Notebook 01/02."
)

# Spatial cells: assign here if Notebook 02 did not already do so.
if spatial.COL_CELL_ID not in traj_df.columns:
    traj_df = spatial.assign_spatial_cells(traj_df, cell_size_meters=config.GRID_CELL_SIZE_METERS)

n_cells = traj_df[spatial.COL_CELL_ID].nunique()
print(f"Grid cell size: {config.GRID_CELL_SIZE_METERS}m | Occupied cells: {n_cells:,}")


Grid cell size: 500.0m | Occupied cells: 5,413


## 4. Build taxi-day sequences

One sequence per (taxi_id, calendar date) — the project's primary
sequence unit, unchanged. `split_by_date=True` (the default) must remain
set for any analysis meant to be comparable to this notebook's results.


In [4]:
labeled_df = sequential.add_temporal_labels(traj_df)
sequences = sequential.build_taxi_sequences(labeled_df, split_by_date=True)

n_sequences = len(sequences)
print(f"Taxi-day sequences built: {n_sequences:,}")

example_key = next(iter(sequences))
print(f"Example key format: {example_key!r}  (taxi_id::date)")


Taxi-day sequences built: 630
Example key format: '107::2008-02-02'  (taxi_id::date)


## 5. Validate sequence count

In [5]:
# Every key must be "taxi_id::date" (exactly one '::' separator) — this
# guards against ever regressing to grouping by taxi_id alone, which
# previously caused enriched-sequence counts to collapse (630 -> 99).
bad_keys = [k for k in sequences if k.count("::") != 1]
assert not bad_keys, f"Found {len(bad_keys)} sequence keys not in 'taxi_id::date' form: {bad_keys[:5]}"

print(f"All {n_sequences:,} sequence keys are in 'taxi_id::date' form.")


All 630 sequence keys are in 'taxi_id::date' form.


## 6. Mine PrefixSpan patterns — main analysis (5% support, max length 4)

`min_support_count` below is shown for transparency; `sequential.mine_patterns()`
computes it internally via `math.ceil(min_support * n_sequences)`, not
`round()`.


In [6]:
min_support_count = math.ceil(config.DEFAULT_MIN_SUPPORT * n_sequences)
print(
    f"n_sequences = {n_sequences} | min_support = {config.DEFAULT_MIN_SUPPORT} "
    f"| min_support_count = ceil({config.DEFAULT_MIN_SUPPORT} * {n_sequences}) = {min_support_count}"
)

patterns_df = sequential.mine_patterns(
    sequences,
    min_support=config.DEFAULT_MIN_SUPPORT,
    max_pattern_length=config.MAX_PATTERN_LENGTH,
)

print(f"Frequent PrefixSpan patterns found: {len(patterns_df):,}")
assert (patterns_df["pattern_length"] <= config.MAX_PATTERN_LENGTH).all(), "max_pattern_length was exceeded"
assert (patterns_df["support_count"] >= min_support_count).all(), "a pattern below the support floor slipped through"


n_sequences = 630 | min_support = 0.05 | min_support_count = ceil(0.05 * 630) = 32
Frequent PrefixSpan patterns found: 22


## 7. Display primary frequent patterns

In [7]:
analysis = sequential.analyze_patterns(patterns_df, top_n=20)
top_patterns = analysis["top_patterns"]
top_patterns[["pattern_str", "support_count", "support_percentage", "pattern_length"]]


,pattern_str,support_count,support_percentage,pattern_length
0,CELL_02242 -> CELL_02242,43,6.825397,2
1,CELL_03758 -> CELL_03757,42,6.666667,2
2,CELL_03719 -> CELL_03758,40,6.349206,2
3,CELL_01519 -> CELL_01520,40,6.349206,2
4,CELL_03719 -> CELL_03757,38,6.031746,2
5,CELL_01743 -> CELL_01746,38,6.031746,2
6,CELL_01968 -> CELL_01967,37,5.873016,2
7,CELL_02612 -> CELL_02611,36,5.714286,2
8,CELL_03584 -> CELL_03758,35,5.555556,2
9,CELL_03677 -> CELL_03719,35,5.555556,2


In [8]:
analysis["length_distribution"]


,pattern_length,n_patterns
0,2,22


## 8. Support sensitivity analysis

**This is a sensitivity analysis only — the primary result remains the
5% analysis above.** Every threshold below is still bounded by
`MAX_PATTERN_LENGTH = 4` and uses the same `ceil()`-based count and the
same original `n_sequences` denominator; no unbounded PrefixSpan run is
performed.


In [9]:
sensitivity_rows = []
for threshold in config.SUPPORT_THRESHOLDS_TO_COMPARE:
    threshold_count = math.ceil(threshold * n_sequences)
    threshold_patterns = sequential.mine_patterns(
        sequences, min_support=threshold, max_pattern_length=config.MAX_PATTERN_LENGTH
    )
    sensitivity_rows.append({
        "min_support": threshold,
        "min_support_count": threshold_count,
        "n_patterns": len(threshold_patterns),
        "is_primary_analysis": threshold == config.DEFAULT_MIN_SUPPORT,
    })

sensitivity_df = pd.DataFrame(sensitivity_rows)
sensitivity_df


,min_support,min_support_count,n_patterns,is_primary_analysis
0,0.01,7,39785,False
1,0.02,13,3426,False
2,0.05,32,22,True


The lowest threshold(s) above are expected to surface many more candidate
patterns than the 5% primary analysis — this is a property of the support
axis, not an error, and none of those lower-threshold results should be
read as the project's headline finding.


## 9. Time-segment analysis

Each time segment gets its **own** taxi-day sequence set and its own
`ceil()`-based minimum support count — segments are not comparable on raw
pattern counts alone because their denominators differ.


In [10]:
time_segment_rows = []
time_segment_patterns = {}

for segment in sorted(config.TIME_SEGMENTS):
    segment_df = labeled_df[labeled_df["time_segment"] == segment]
    if segment_df.empty:
        continue
    segment_sequences = sequential.build_taxi_sequences(segment_df, split_by_date=True)
    n_seg = len(segment_sequences)
    if n_seg == 0:
        time_segment_rows.append({
            "time_segment": segment, "n_sequences": 0,
            "min_support_count": None, "n_patterns_len_ge_2": 0,
        })
        continue

    seg_min_count = math.ceil(config.DEFAULT_MIN_SUPPORT * n_seg)
    try:
        seg_patterns = sequential.mine_patterns(
            segment_sequences, min_support=config.DEFAULT_MIN_SUPPORT,
            max_pattern_length=config.MAX_PATTERN_LENGTH,
        )
    except ValueError:
        seg_patterns = sequential._empty_patterns()

    time_segment_patterns[segment] = seg_patterns
    time_segment_rows.append({
        "time_segment": segment,
        "n_sequences": n_seg,
        "min_support_count": seg_min_count,
        "n_patterns_len_ge_2": len(seg_patterns),
    })

time_segment_summary = pd.DataFrame(time_segment_rows)
time_segment_summary


,time_segment,n_sequences,min_support_count,n_patterns_len_ge_2
0,afternoon,581,30,0
1,evening,506,26,0
2,morning,411,21,0
3,night,477,24,0


A time segment showing `n_patterns_len_ge_2 == 0` at 5% support is an
honest result, not a bug — it means no ordered cell sequence recurred in
at least `min_support_count` taxi-days within that segment alone. Patterns
are never manufactured to avoid a zero.


## 10. Weekday/weekend comparison

Weekday and weekend sequences have **different denominators** — always
report both `support_count` and `support_percentage`, and read differences
in the *number* of unique patterns with that in mind rather than treating
it as a direct measure of "more mobility variety".


In [11]:
weekday_df = labeled_df[labeled_df["day_type"] == "weekday"]
weekend_df = labeled_df[labeled_df["day_type"] == "weekend"]

weekday_sequences = sequential.build_taxi_sequences(weekday_df, split_by_date=True)
weekend_sequences = sequential.build_taxi_sequences(weekend_df, split_by_date=True)

n_weekday = len(weekday_sequences)
n_weekend = len(weekend_sequences)
weekday_min_count = math.ceil(config.DEFAULT_MIN_SUPPORT * n_weekday)
weekend_min_count = math.ceil(config.DEFAULT_MIN_SUPPORT * n_weekend)

print(f"weekday: n_sequences={n_weekday}, min_support_count=ceil(0.05*{n_weekday})={weekday_min_count}")
print(f"weekend: n_sequences={n_weekend}, min_support_count=ceil(0.05*{n_weekend})={weekend_min_count}")

weekday_patterns = (
    sequential.mine_patterns(weekday_sequences, config.DEFAULT_MIN_SUPPORT, config.MAX_PATTERN_LENGTH)
    if n_weekday else sequential._empty_patterns()
)
weekend_patterns = (
    sequential.mine_patterns(weekend_sequences, config.DEFAULT_MIN_SUPPORT, config.MAX_PATTERN_LENGTH)
    if n_weekend else sequential._empty_patterns()
)

from pattern_ranking import compare_pattern_sets
comparison = compare_pattern_sets(weekday_patterns, weekend_patterns, "weekday", "weekend")

print(
    f"unique to weekday: {len(comparison['unique_to_a'])} | "
    f"unique to weekend: {len(comparison['unique_to_b'])} | "
    f"common: {len(comparison['common'])}"
)


weekday: n_sequences=446, min_support_count=ceil(0.05*446)=23
weekend: n_sequences=184, min_support_count=ceil(0.05*184)=10
unique to weekday: 1 | unique to weekend: 236 | common: 8


**Interpretation note.** Weekday and weekend pattern counts are not
directly comparable as raw numbers: they come from different-sized taxi-day
samples (different `min_support_count` floors), so a larger raw count of
"unique" patterns on one side reflects a lower absolute support
requirement on that side as much as it reflects any real behavioral
difference. `support_percentage` (not raw counts) is the number to compare
across the two groups.


## 11. Direct/contiguous transition analysis (supplementary)

This is a **separate analysis from PrefixSpan** and must not be presented
as PrefixSpan patterns: a PrefixSpan pattern `A -> B` means A occurred
before B somewhere in the sequence (gaps allowed); a direct transition
`A -> B` means A and B were immediately consecutive cells (after
consecutive-duplicate compression).


In [12]:
direct_patterns = sequential.mine_contiguous_patterns(
    sequences,
    min_support=config.DEFAULT_MIN_SUPPORT,
    max_pattern_length=2,
    exclude_repeated_cells=True,
)
print(f"Direct/contiguous transitions @5% support: {len(direct_patterns)}")
direct_patterns.head(10)


Direct/contiguous transitions @5% support: 2


,pattern,support_count,support_percentage,pattern_length,mining_method,is_direct_transition
0,"(CELL_01519, CELL_01520)",34,5.396825,2,contiguous_ngram,True
1,"(CELL_03758, CELL_03757)",32,5.079365,2,contiguous_ngram,True


## 12. Interpretation and limitations

- **Support denominator.** All support figures in this notebook use the
  count of taxi-day sequences as the denominator — never GPS-point count,
  distinct-taxi count, or the post-pruning sequence count.
- **PrefixSpan vs. direct transitions.** Patterns in Sections 6-10 are
  ordered subsequences (gaps allowed); Section 11's direct transitions are
  a distinct, adjacency-only analysis and are not mixed into the PrefixSpan
  counts anywhere in this notebook.
- **Repeated-cell patterns** (e.g. `CELL_X -> CELL_X`) are valid PrefixSpan
  output (they represent a revisit, e.g. `A -> X -> Y -> A` contains
  `A -> A`) and are kept, but are never described as a direct movement
  transition.
- **Zero-pattern subgroups** (Section 9) are reported as a genuine result,
  not treated as evidence of a pipeline error.
- **Sample size.** This notebook runs on the project's development sample,
  not the full T-Drive release; absolute pattern counts will change (and
  should be re-validated) on a full run.


## 13. Save Notebook 03 outputs

In [13]:
config.ensure_output_dirs()

patterns_out = config.TABLES_DIR / "nb03_frequent_patterns.csv"
sensitivity_out = config.TABLES_DIR / "nb03_support_sensitivity.csv"
time_segment_out = config.TABLES_DIR / "nb03_time_segment_summary.csv"
direct_out = config.TABLES_DIR / "nb03_direct_transitions.csv"

patterns_df.assign(pattern=patterns_df["pattern"].apply(" -> ".join)).to_csv(patterns_out, index=False)
sensitivity_df.to_csv(sensitivity_out, index=False)
time_segment_summary.to_csv(time_segment_out, index=False)
direct_patterns.assign(pattern=direct_patterns["pattern"].apply(" -> ".join)).to_csv(direct_out, index=False) if len(direct_patterns) else None

print("Saved:")
for p in [patterns_out, sensitivity_out, time_segment_out, direct_out]:
    print(f"  - {p}")


Saved:
  - C:\Users\user\Downloads\urban-taxi-mobility\urban-taxi-mobility\outputs\tables\nb03_frequent_patterns.csv
  - C:\Users\user\Downloads\urban-taxi-mobility\urban-taxi-mobility\outputs\tables\nb03_support_sensitivity.csv
  - C:\Users\user\Downloads\urban-taxi-mobility\urban-taxi-mobility\outputs\tables\nb03_time_segment_summary.csv
  - C:\Users\user\Downloads\urban-taxi-mobility\urban-taxi-mobility\outputs\tables\nb03_direct_transitions.csv
